# Closed-Loop Evaluation (Colab)

This notebook evaluates completed Closed-Loop simulation artifacts (`csv` + `json`) saved in Google Drive.

It uses reusable utilities in `src/eval` to:
- discover and load run artifacts,
- compute method summaries,
- estimate conditional value of surprise beyond risk,
- compute compute-normalized blind-spot discovery efficiency.


## Usage

1. Set your `RUN_TAG` and `PERSIST_ROOT`.
2. Run discovery to select merged or shard outputs.
3. Run analysis cells.
4. Export evaluation tables back to Drive.


In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/achyutmorang/thesis-research-colab.git"
REPO_DIR = "/content/thesis-research-colab"

if os.path.exists(REPO_DIR):
    print("Repo exists, fast-forwarding main...")
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "main"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())


In [ ]:
# Mount Google Drive in Colab
import os

if os.environ.get('COLAB_RELEASE_TAG'):
    try:
        from google.colab import drive
        if not os.path.ismount('/content/drive'):
            drive.mount('/content/drive')
        else:
            print('[drive] /content/drive already mounted')
    except Exception as e:
        print('[drive] mount skipped:', e)
else:
    print('[drive] non-Colab environment; skipping mount')


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from src.eval import (
    budget_normalized_efficiency,
    conditional_lift_by_risk_bins,
    discover_run_prefixes,
    discovery_auc,
    discovery_curve_from_trace,
    load_run_artifacts,
    method_summary,
    select_default_run_prefix,
)

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 200)


In [ ]:
# ===== User config =====
RUN_TAG = "closedloop_v1"
PERSIST_ROOT = "/content/drive/MyDrive/closedloop_runs/v1"
N_SHARDS = 5
OUT_RUN_TAG = None             # e.g. "closedloop_v1_merged"
PREFER_KIND = "merged"        # "merged" or "shard" or "base"
REQUIRE_TRACE = True

# Optional export directory (inside persist root)
EXPORT_DIR = f"{PERSIST_ROOT}/evaluation"


In [ ]:
candidates_df = discover_run_prefixes(
    run_tag=RUN_TAG,
    persist_root=PERSIST_ROOT,
    n_shards=N_SHARDS,
    out_run_tag=OUT_RUN_TAG,
)

display(candidates_df)

selected_run_prefix = select_default_run_prefix(candidates_df, prefer_kind=PREFER_KIND)
print("selected_run_prefix:", selected_run_prefix)


In [ ]:
artifacts = load_run_artifacts(selected_run_prefix, require_trace=REQUIRE_TRACE)

results_df = artifacts.results_df.copy()
trace_df = artifacts.trace_df.copy()
thresholds = artifacts.thresholds

print("results rows:", len(results_df))
print("trace rows:", len(trace_df))
print("threshold keys:", sorted(thresholds.keys())[:12], "...")

display(results_df.head())
if len(trace_df) > 0:
    display(trace_df.head())


In [ ]:
summary_df = method_summary(results_df)
eff_df = budget_normalized_efficiency(results_df)

print("Method summary")
display(summary_df)

print("Budget-normalized efficiency")
display(eff_df)


In [ ]:
# Conditional value of surprise beyond risk:
# treatment (default joint) vs control (default risk_only) within risk bins.

cond_bins_df, cond_overall_df = conditional_lift_by_risk_bins(
    results_df,
    treatment="joint",
    control="risk_only",
    risk_col="risk_sks",
    outcome_col="blind_spot_proxy_hit",
    n_bins=10,
    min_bin_count=5,
    bootstrap_samples=2000,
    bootstrap_seed=17,
)

print("Overall paired effect")
display(cond_overall_df)

print("Risk-bin conditional lift")
display(cond_bins_df)

if len(cond_bins_df) > 0:
    plt.figure(figsize=(8, 4))
    sns.barplot(data=cond_bins_df, x="risk_bin", y="lift", color="#1f77b4")
    plt.xticks(rotation=45, ha='right')
    plt.title("Conditional Blind-Spot Lift (joint - risk_only) by Risk Bin")
    plt.ylabel("Lift in blind_spot_proxy_hit rate")
    plt.xlabel("Risk bin (quantiles)")
    plt.tight_layout()
    plt.show()


In [ ]:
# Compute-normalized discovery efficiency from per-eval trace.

if len(trace_df) == 0:
    print("No per_eval_trace rows found. Re-run with trace export enabled.")
    curve_df = pd.DataFrame()
    auc_df = pd.DataFrame()
else:
    curve_df = discovery_curve_from_trace(
        trace_df=trace_df,
        thresholds=thresholds,
    )
    auc_df = discovery_auc(curve_df)

    display(auc_df)

    plt.figure(figsize=(8, 5))
    sns.lineplot(data=curve_df, x="eval_index", y="discovery_rate", hue="method", marker="o")
    plt.title("Blind-Spot Discovery vs Evaluation Budget")
    plt.xlabel("Evaluation index (compute budget)")
    plt.ylabel("Discovery rate (ever-hit up to eval index)")
    plt.tight_layout()
    plt.show()


In [ ]:
from pathlib import Path

Path(EXPORT_DIR).mkdir(parents=True, exist_ok=True)
prefix_name = Path(selected_run_prefix).name

summary_path = f"{EXPORT_DIR}/{prefix_name}_eval_method_summary.csv"
eff_path = f"{EXPORT_DIR}/{prefix_name}_eval_budget_efficiency.csv"
cond_bins_path = f"{EXPORT_DIR}/{prefix_name}_eval_conditional_lift_bins.csv"
cond_overall_path = f"{EXPORT_DIR}/{prefix_name}_eval_conditional_lift_overall.csv"
curve_path = f"{EXPORT_DIR}/{prefix_name}_eval_discovery_curve.csv"
auc_path = f"{EXPORT_DIR}/{prefix_name}_eval_discovery_auc.csv"

summary_df.to_csv(summary_path, index=False)
eff_df.to_csv(eff_path, index=False)
cond_bins_df.to_csv(cond_bins_path, index=False)
cond_overall_df.to_csv(cond_overall_path, index=False)
if 'curve_df' in globals() and isinstance(curve_df, pd.DataFrame):
    curve_df.to_csv(curve_path, index=False)
if 'auc_df' in globals() and isinstance(auc_df, pd.DataFrame):
    auc_df.to_csv(auc_path, index=False)

print("Saved:")
for p in [summary_path, eff_path, cond_bins_path, cond_overall_path, curve_path, auc_path]:
    print(" -", p)


## Notes

- `blind_spot_proxy_hit` is the current Closed-Loop proxy label from simulation outputs.
- For strict publication claims, pair these tables with pre-registered thresholds and paired-seed significance reporting.
